Verify NVIDIA GPU Availability

In [1]:
!nvidia-smi

Fri Mar 20 18:23:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Unzip images to a custom data folder

In [2]:
!unzip -q /content/data.zip -d /content/custom_data

unzip:  cannot find or open /content/data.zip, /content/data.zip.zip or /content/data.zip.ZIP.


Split images into train and validation folders

In [3]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

!python train_val_split.py --datapath="/content/custom_data/data/images" --train_pct=0.9

--2026-03-20 18:23:54--  https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3203 (3.1K) [text/plain]
Saving to: ‘/content/train_val_split.py’

/content/train_val_ 100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-03-20 18:23:54 (53.4 MB/s) - ‘/content/train_val_split.py’ saved [3203/3203]

Directory specified by --datapath not found. Verify the path is correct (and uses double back slashes if on Windows) and try again.


In [4]:
import os
import glob
import random
import shutil

source_path = '/content/custom_data/data'
train_path = '/content/yolo_data/train'
val_path = '/content/yolo_data/val'

for p in [train_path, val_path]:
    os.makedirs(os.path.join(p, 'images'), exist_ok=True)
    os.makedirs(os.path.join(p, 'labels'), exist_ok=True)

image_exts = ['*.png', '*.jpg', '*.jpeg', '*.JPG', '*.PNG']
images = []
for ext in image_exts:
    images.extend(glob.glob(os.path.join(source_path, 'images', ext)))

random.shuffle(images)
split_idx = int(len(images) * 0.9)
train_files = images[:split_idx]
val_files = images[split_idx:]

def move_files(file_list, target_dir):
    for img_path in file_list:
        file_name = os.path.basename(img_path)
        base_name = os.path.splitext(file_name)[0]

        shutil.copy(img_path, os.path.join(target_dir, 'images', file_name))

        label_path = os.path.join(source_path, 'labels', base_name + '.txt')
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(target_dir, 'labels', base_name + '.txt'))

move_files(train_files, train_path)
move_files(val_files, val_path)

print(f"Done! Sent {len(train_files)} to training and {len(val_files)} to validation.")

Done! Sent 0 to training and 0 to validation.


install the Ultralytics

In [5]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.5 MB/s eta 0:00:00


Configure Training

In [6]:
import yaml

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found at {path_to_classes_txt}')
    return

  with open(path_to_classes_txt, 'r') as f:
    classes = [line.strip() for line in f.readlines() if line.strip()]

  number_of_classes = len(classes)

  data = {
      'path': '/content/yolo_data',
      'train': 'train/images',
      'val': 'val/images',
      'nc': number_of_classes,
      'names': classes
  }

  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

  print(f'Created config file at {path_to_data_yaml}')

path_to_classes_txt = '/content/custom_data/data/classes.txt'
path_to_data_yaml = '/content/data.yaml'
create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nNew File contents:\n')
!cat /content/data.yaml

classes.txt file not found at /content/custom_data/data/classes.txt

New File contents:

cat: /content/data.yaml: No such file or directory


 Run Training

In [7]:
!yolo detect train data=/content/data.yaml model=yolov8s.pt epochs=40 imgsz=640

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, i

Test Model

In [8]:
!yolo detect predict model=/content/runs/detect/train2/weights/best.pt source=/content/yolo_data/val/images save=True

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 956, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 283, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 1515, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^^

In [9]:
import glob
import os
from IPython.display import Image, display

predict_folders = glob.glob('/content/runs/detect/predict*/')
if predict_folders:
    latest_folder = max(predict_folders, key=os.path.getmtime)
    print(f"Showing results from: {latest_folder}")

    for image_path in glob.glob(f'{latest_folder}/*.jpg')[:10]:
        display(Image(filename=image_path, height=400))
        print('\n')
else:
    print("No prediction folders found!")

No prediction folders found!


Download YOLO Model

In [10]:
!mkdir -p /content/my_model

!cp /content/runs/detect/train2/weights/best.pt /content/my_model/pothole_best_weights.pt
!cp -r /content/runs/detect/train2 /content/my_model/train_results

!zip -r /content/pothole_model_package.zip /content/my_model

cp: cannot stat '/content/runs/detect/train2/weights/best.pt': No such file or directory
cp: cannot stat '/content/runs/detect/train2': No such file or directory
  adding: content/my_model/ (stored 0%)


In [11]:
from google.colab import files
files.download('/content/pothole_model_package.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>